# 02 · Multimodal Model Training & Inference Pipeline

> **PAI DSW 单卡 NVIDIA A10 24GB · Qwen2-VL-2B-Instruct · 4-bit NF4 + FP16 · QLoRA · 同图推理对比**

这份 Notebook **完全自包含**：下面所有数据准备、模型加载、LoRA、训练与推理函数均以独立 Code Cell 给出，不依赖 Notebook 01 或仓库中的业务 `.py` 文件。只依赖 `.env`（可选）、安装的 Python 第三方库、先前导出的本地数据或 Paimon 表。

建议：已完成本项目上次实验的用户**优先复用已有的 19/1 JSONL、Baseline 和 LoRA Adapter**，避免重新划分验证集、重复扣费和覆盖权重。

**重要限制**：19 张连续帧训练 + 1 张相似帧验证，仅是工程流程/Caption 风格演示，不能证明泛化能力。

## Step 0 · 环境准备 / Restart Kernel

在 DSW Terminal 中克隆仓库，编辑项目根目录 `.env` 并安装 `requirements-data.txt`、`requirements-train.txt` 和 `constraints-cu126.txt`（详见 README / Notebook 01 Step 0）。**安装后重启 Notebook Kernel**。

已验证组合：**NVIDIA A10 24GB / PyTorch 2.9.1+cu126 / transformers 4.52.4 / peft 0.15.2 / bitsandbytes 0.48.0**；使用 FP16、eager attention，不使用 BF16 / FlashAttention-2。不要运行可能替换 PyTorch 的 `pip --force-reinstall pypaimon`。

## Step 1 · 加载 `.env` 并检查 A10 GPU

In [ ]:
# Step 1：Notebook 自包含；查找仓库根目录，仅用 .env.example 确认目录
import os
import sys
import shutil
import json
import importlib.metadata
import platform
from pathlib import Path
from dataclasses import dataclass
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents,
              cwd/'aliyun-autonomous-driving-multimodal-pipeline',
              Path.home()/'workspace'/'aliyun-autonomous-driving-multimodal-pipeline']
ROOT = next((p for p in candidates if (p/'README.md').is_file()
             and (p/'.env.example').is_file() and (p/'notebooks').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('请在已 git clone 的仓库根目录或 notebooks/ 目录运行本 Notebook。')
load_dotenv(ROOT/'.env', override=False)
print('仓库:', ROOT)
print('.env 文件存在:', (ROOT/'.env').is_file())

In [ ]:
# Step 2：所有参数通过环境变量 / .env 加载；不在 Notebook 内硬编码密钥
@dataclass(frozen=True)
class Settings:
    region: str = os.getenv("ALIYUN_REGION", "cn-hangzhou")
    dlf_rest_uri: str = os.getenv("DLF_REST_URI", "https://dlfnext.cn-hangzhou.aliyuncs.com")
    dlf_warehouse: str = os.getenv("DLF_WAREHOUSE", "vehicle_lake_demo")
    oss_endpoint: str = os.getenv("DLF_OSS_ENDPOINT", "oss-cn-hangzhou.aliyuncs.com")
    access_key_id: str = os.getenv("ALIBABA_CLOUD_ACCESS_KEY_ID", "")
    access_key_secret: str = os.getenv("ALIBABA_CLOUD_ACCESS_KEY_SECRET", "")
    security_token: str = os.getenv("ALIBABA_CLOUD_SECURITY_TOKEN", "")
    raw_table: str = os.getenv("PAIMON_RAW_TABLE", "default.camfront_raw_images")
    enriched_table: str = os.getenv("PAIMON_ENRICHED_TABLE", "default.camfront_ai_enriched")
    vision_base_url: str = os.getenv("VISION_API_BASE_URL", "")
    vision_key: str = os.getenv("VISION_API_KEY", "")
    vision_model: str = os.getenv("VISION_MODEL", "your-vision-model")
    embedding_url: str = os.getenv("EMBEDDING_API_URL", "")
    embedding_key: str = os.getenv("EMBEDDING_API_KEY", "")
    embedding_model: str = os.getenv("EMBEDDING_MODEL", "tongyi-embedding-vision-plus")
    vector_dim: int = int(os.getenv("EMBEDDING_VECTOR_DIM", "1152"))
    work_dir: Path = Path(os.getenv("WORK_DIR", "/mnt/systemDisk/ad_multimodal"))
    model_id: str = os.getenv("MODEL_ID", "Qwen/Qwen2-VL-2B-Instruct")
    model_cache_dir: Path = Path(os.getenv("MODEL_CACHE_DIR", "/mnt/systemDisk/model_cache"))
    seed: int = int(os.getenv("SPLIT_RANDOM_SEED", "20260520"))
    # 验证集比例，默认 0.05，即 5%
    val_ratio: float = float(os.getenv("VALIDATION_RATIO", "0.05"))
    image_max_tokens: int = int(os.getenv("IMAGE_MAX_TOKENS", "256"))

    @property
    def dataset_dir(self) -> Path:
        return self.work_dir / "vlm_training"

    @property
    def lora_output_dir(self) -> Path:
        return self.work_dir / "qwen2vl_lora"

    @property
    def nu_scenes_dir(self) -> Path:
        return self.work_dir / "nuscenes"

    @property
    def caption_csv(self) -> Path:
        return self.work_dir / "caption_metadata_results.csv"

    @property
    def embedding_cache(self) -> Path:
        return self.work_dir / "embedding_cache" / self.embedding_model

    def require_dlf_credentials(self) -> None:
        if not self.access_key_id or not self.access_key_secret:
            raise RuntimeError("Set ALIBABA_CLOUD_ACCESS_KEY_ID / ALIBABA_CLOUD_ACCESS_KEY_SECRET (or adapt a supported role credential provider).")

    def catalog_options(self) -> dict[str, str]:
        """DLF 2.0 REST catalog options used in the verified PyPaimon 1.4 flow."""
        self.require_dlf_credentials()
        options = {
            "metastore": "rest", "uri": self.dlf_rest_uri,
            "warehouse": self.dlf_warehouse,
            "dlf.region": self.region,
            "token.provider": "dlf",
            "dlf.access-key-id": self.access_key_id,
            "dlf.access-key-secret": self.access_key_secret,
            "dlf.oss-endpoint": self.oss_endpoint,
        }
        if self.security_token:
            # Validate against the deployed PyPaimon version before using STS.
            options["dlf.security-token"] = self.security_token
        return options

settings = Settings()
def check_environment() -> bool:
    """Print diagnostics without revealing secrets or changing packages."""
    import importlib.metadata
    import platform
    print("Python:", platform.python_version())
    for name in ("numpy", "pandas", "pypaimon", "pyarrow", "torch", "transformers", "peft", "bitsandbytes", "modelscope"):
        try:
            print(f"{name:15} {importlib.metadata.version(name)}")
        except importlib.metadata.PackageNotFoundError:
            print(f"{name:15} [not installed]")
    print("Work dir:", settings.work_dir)
    print("Work-dir parent exists:", settings.work_dir.parent.exists())
    if settings.work_dir.parent.exists():
        print("Free GiB:", round(shutil.disk_usage(settings.work_dir.parent).free / 2**30, 1))
    for label, value in (("DLF credentials", bool(settings.access_key_id and settings.access_key_secret)),
                         ("Vision API", bool(settings.vision_base_url and settings.vision_key)),
                         ("Embedding API", bool(settings.embedding_url and settings.embedding_key))):
        print(f"{label}: {'configured' if value else 'NOT configured'}")
    try:
        import torch
        print("PyTorch CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            sm = torch.cuda.get_device_capability(0)
            print("GPU:", name, "compute capability:", sm)
            x = torch.randn(64, 64, device="cuda")
            torch.cuda.synchronize()
            print("GPU tensor test:", float((x @ x).mean()))
            return True
    except (ImportError, RuntimeError) as exc:
        print("GPU check failed:", str(exc)[:300])
    return False

check_environment()
print('Paimon Raw / Enriched:', settings.raw_table, settings.enriched_table)
print('Working dir:', settings.work_dir)

## Step 2 · 从本地 JSONL 复用 19/1 划分，或从 Paimon 重新导出

建议优先复用先前 Notebook 中的 `vlm_training/train.jsonl` / `validation.jsonl` 和 `baseline_prediction.json`；重新划分可能导致验证图像改变，无法公平比较。只有首次准备数据才将 `RUN_EXPORT_FROM_PAIMON=True`。

In [ ]:
# 训练数据处理完整实现：有效行筛选 → 稳定划分 → 数据导出 → 旧数据校验
import hashlib
import uuid
import json
import pandas as pd
from PIL import Image
from IPython.display import display
def select_valid_rows(df: pd.DataFrame, dataset_tag: int = 1) -> pd.DataFrame:
    required = {"id", "fileName", "picture", "caption", "dataset_tag"}
    if not required <= set(df.columns):
        raise ValueError(f"Missing Paimon columns: {sorted(required - set(df.columns))}")
    rows = df.loc[
        df["dataset_tag"].eq(dataset_tag)
        & df["picture"].notna()
        & df["caption"].notna()
        & df["caption"].astype(str).str.strip().ne("")
    ].copy()
    if rows.empty:
        raise ValueError("No labeled rows with dataset_tag=1")
    if rows["id"].duplicated().any() or rows["fileName"].duplicated().any():
        raise ValueError("Duplicate image id or filename")
    return rows.reset_index(drop=True)


def split_records(rows: pd.DataFrame, val_ratio: float, seed: int):
    """Stable selection by image ID before seeded shuffle; reproduces across read orders."""
    if not 0 < val_ratio < 1 or len(rows) < 2:
        raise ValueError("Need >=2 samples and 0<val_ratio<1")
    ordered = rows.sort_values("id").sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = max(1, min(len(rows)-1, round(len(rows)*val_ratio)))
    validation = ordered.iloc[:n_val].copy()
    training = ordered.iloc[n_val:].copy()
    assert set(training["id"]).isdisjoint(set(validation["id"]))
    return training, validation


def dataset_statistics(rows: pd.DataFrame) -> dict:
    """Dataset Statistics Overview from the tutorial: caption and image sizes."""
    captions = rows["caption"].astype(str).str.len()
    image_sizes = rows["picture"].map(lambda b: len(bytes(b)))
    result = {
        "samples": int(len(rows)),
        "caption_chars_min": int(captions.min()),
        "caption_chars_max": int(captions.max()),
        "caption_chars_mean": round(float(captions.mean()), 2),
        "caption_chars_median": round(float(captions.median()), 2),
        "image_size_kb_min": round(float(image_sizes.min()) / 1024, 2),
        "image_size_kb_max": round(float(image_sizes.max()) / 1024, 2),
        "image_size_kb_mean": round(float(image_sizes.mean()) / 1024, 2),
    }
    return result


def write_dataset(train: pd.DataFrame, val: pd.DataFrame, dest: Path, seed: int) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    image_dir = dest / "images"
    image_dir.mkdir(exist_ok=True)
    signature = hashlib.sha256(
        "\n".join(sorted([str(x) for x in list(train["id"]) + list(val["id"])] )).encode()
    ).hexdigest()
    manifest_file = dest / "split_manifest.json"
    manifest = {
        "seed": seed,
        "all_ids_sha256": signature,
        "train_ids": [str(x) for x in train["id"]],
        "validation_ids": [str(x) for x in val["id"]],
    }
    if not manifest_file.exists() and any((dest / name).exists() for name in ("train.jsonl", "validation.jsonl")):
        raise RuntimeError("Existing JSONL data has no split manifest; use a new --output-dir to protect the previous validation sample and baseline")
    if manifest_file.exists():
        old = json.loads(manifest_file.read_text(encoding="utf-8"))
        if old != manifest:
            raise RuntimeError("Existing split_manifest.json differs. Choose a new --output-dir to avoid overwriting a validated split.")
    else:
        manifest_file.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    for subset_name, frame in (("train", train), ("validation", val)):
        out = []
        for row in frame.to_dict("records"):
            name = str(row["id"]) + ".jpg"
            target = image_dir / name
            data = bytes(row["picture"])
            if target.exists():
                if target.read_bytes() != data:
                    raise ValueError(f"Image file mismatch: {target}")
            else:
                target.write_bytes(data)
            with Image.open(target) as im:
                im.verify()
            out.append({"id": str(row["id"]), "image": f"images/{name}",
                        "caption": str(row["caption"]), "fileName": str(row["fileName"])})
        with (dest / f"{subset_name}.jsonl").open("w", encoding="utf-8") as f:
            for entry in out:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"{subset_name}: {len(out)} samples")


def load_records(path: Path) -> list[dict]:
    base = path.parent
    with path.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    for row in records:
        if "id" not in row:
            if not row.get("fileName"):
                raise ValueError("Legacy JSONL requires fileName to reconstruct image ID")
            row["id"] = str(uuid.uuid5(uuid.NAMESPACE_URL, str(row["fileName"])))
        image_path = Path(row["image"])
        if not image_path.is_absolute():
            image_path = base / image_path
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        row["image"] = str(image_path)
    return records


def verify_existing_dataset(dest: Path) -> dict:
    """Read earlier Notebook's 19/1 JSONL without changing its split or captions."""
    train = load_records(dest / "train.jsonl")
    val = load_records(dest / "validation.jsonl")
    if not train or not val:
        raise ValueError("Both training and validation sets must be non-empty")
    train_ids = [r["id"] for r in train]
    val_ids = [r["id"] for r in val]
    if len(train_ids + val_ids) != len(set(train_ids + val_ids)):
        raise ValueError("Duplicate IDs across the split")
    if (dest / "baseline_prediction.json").exists():
        baseline = json.loads((dest / "baseline_prediction.json").read_text(encoding="utf-8"))
        if Path(baseline["image"]).resolve() != Path(val[0]["image"]).resolve():
            raise ValueError("Existing baseline image is not the held-out validation image")
        if baseline.get("reference_caption") != val[0]["caption"]:
            raise ValueError("Baseline reference caption does not match validation JSONL")
    manifest_path = dest / "split_manifest.json"
    if not manifest_path.exists():
        info = {
            "legacy_imported": True,
            "note": "Original Notebook split preserved; do NOT regenerate from sorted Paimon IDs",
            "train_ids": train_ids,
            "validation_ids": val_ids,
        }
        manifest_path.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Verified original split: train={len(train)}, validation={len(val)}, manifest={manifest_path}")
    return {"train": len(train), "validation": len(val)}

# 仅在首次需要从 Paimon 导出数据时使用以下两个本地定义
def connect_catalog():
    from pypaimon import CatalogFactory
    return CatalogFactory.create(settings.catalog_options())


def read_table(table):
    import pandas as pd
    rb = table.new_read_builder()
    splits = rb.new_scan().plan().splits()
    return rb.new_read().to_pandas(splits) if splits else pd.DataFrame()

DATA_DIR = settings.dataset_dir
RUN_EXPORT_FROM_PAIMON = False
if (DATA_DIR/'train.jsonl').exists() and (DATA_DIR/'validation.jsonl').exists():
    verify_existing_dataset(DATA_DIR)
elif RUN_EXPORT_FROM_PAIMON:
    source_df = read_table(connect_catalog().get_table(settings.enriched_table))
    selected_df = select_valid_rows(source_df, dataset_tag=1)
    train_df, val_df = split_records(selected_df, settings.val_ratio, settings.seed)
    print('训练样本统计:', dataset_statistics(train_df))
    write_dataset(train_df, val_df, DATA_DIR, settings.seed)
else:
    raise FileNotFoundError('未找到训练 JSONL：请先运行 Notebook 01，或将 RUN_EXPORT_FROM_PAIMON 改 True')

train_records = load_records(DATA_DIR/'train.jsonl')
val_records = load_records(DATA_DIR/'validation.jsonl')
assert len(val_records) == 1, '本案例需要恰好 1 张验证图像；更多验证图像时需修改评估方式'
sample = val_records[0]
print('训练数量:',len(train_records),'验证数量:',len(val_records),'验证文件:',sample['fileName'])

## Step 3 · 可视化样本、Caption 统计与数据集分布

这次 20 张图片来自相似的连续帧，仅用于工程 Smoke Test。

In [ ]:
print('训练样本:',train_records[0]['fileName'])
print('Caption:',train_records[0]['caption'])
with Image.open(train_records[0]['image']) as im:
    display(im.convert('RGB').resize((680,382)))
lengths = [len(r['caption']) for r in train_records]
print('平均 Caption 字符数:',round(sum(lengths)/len(lengths),2))
print('验证 Caption:',sample['caption'])
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,3))
    plt.hist(lengths,bins=min(8,len(lengths)))
    plt.title('Training caption lengths'); plt.grid(alpha=.2); plt.show()
except ImportError:
    print('matplotlib 未安装，文字统计不受影响')

## Step 4 · ModelScope 模型缓存与 NF4/FP16 量化加载

本教程在 A10 上使用 4-bit NF4 存储权重、FP16 计算；不启用 FlashAttention2/BF16。模型载入完成后，Notebook 变量 `model` 与 `processor` 会留在当前 Kernel。

In [ ]:
# 单卡 CUDA 检查、模型下载/缓存、Qwen2-VL 4-bit 加载的完整实现
import torch
def require_cuda() -> None:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available; check NVIDIA GPU and PyTorch cu126 installation")
    print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda,
          "architectures:", torch.cuda.get_arch_list())


def resolve_model_dir(model_dir: str | Path | None = None) -> Path:
    """Prefer explicit local dir; otherwise download/reuse ModelScope cache."""
    if model_dir:
        path = Path(model_dir).expanduser().resolve()
        if not (path / "config.json").is_file():
            raise FileNotFoundError(f"No config.json under {path}")
        return path
    from modelscope import snapshot_download
    cached = Path(snapshot_download(settings.model_id, cache_dir=str(settings.model_cache_dir)))
    print("ModelScope model directory:", cached)
    return cached


def load_base_model(model_dir: str | Path | None = None, *, max_tokens: int | None = None):
    # model_dir 可指定本地模型目录；max_tokens 控制视觉 Token 预算，不是生成文本长度。
    import torch
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration
    # 确认 CUDA 可用，再解析本地目录或下载后的 ModelScope 缓存目录。
    require_cuda()
    path = resolve_model_dir(model_dir)
    tokens = max_tokens or settings.image_max_tokens
    if tokens < 128:
        print("Warning: very low visual token limit may lose scene details")
    # Qwen2-VL 按每个视觉 Token 约 28×28 像素设置图像缩放范围，限制显存开销。
    # Processor 从已解析的本地目录读取，不在此处访问远端下载文件。
    processor = AutoProcessor.from_pretrained(
        path,
        min_pixels=min(128, tokens) * 28 * 28,
        max_pixels=tokens * 28 * 28,
        use_fast=False,
        local_files_only=True,
    )
    # 以 4-bit NF4 量化支持的权重，计算时使用 FP16；双重量化进一步压缩量化参数。
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    # 将整个模型放到第 0 张 GPU；非量化浮点部分使用 FP16，Attention 沿用 eager 实现。
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        path, quantization_config=quantization,
        torch_dtype=torch.float16, device_map={"": 0},
        attn_implementation="eager", local_files_only=True,
    )
    # 这里只统计 PyTorch 当前分配的显存，不等于 GPU 总占用或训练峰值。
    print("Allocated GiB:", round(torch.cuda.memory_allocated() / 2**30, 2))
    return model, processor, path

# 本 Cell 先确认 GPU 并准备模型缓存；实际加载模型需调用 load_base_model。
require_cuda()
MODEL_DIR = resolve_model_dir()
print('模型缓存位置:',MODEL_DIR)

In [ ]:
# 不要在已挂载 LoRA 的同一个 Kernel 中重复加载基础模型
from peft import PeftModel
if 'model' in globals():
    print('已加载模型:',type(model).__name__,'；需要重新加载请重启 Kernel。')
else:
    model,processor,MODEL_DIR = load_base_model(MODEL_DIR)
    print('已加载',type(model).__name__,'分配显存 GiB:',round(torch.cuda.memory_allocated()/2**30,2))

## Step 5 · 基础模型同图推理：固定 Prompt + 解码参数

首次推理保存 `baseline_prediction.json`；后续复用。必须是 **未加载 LoRA 的基础模型** 生成基线。**微调前后使用相同图像、Prompt、最大 Token 数与解码方式**。

In [ ]:
# 自回归生成 Caption 函数
# 固定描述要求，微调前后复用同一提示词，避免因指令变化影响对比。
PROMPT = (
    "Describe this autonomous driving scene in one or two concise "
    "English sentences. Mention the road layout, traffic participants, "
    "weather and clearly visible risks. Do not use bullet points or speculate."
)


# max_new_tokens 限制新生成的文本 Token 数，与视觉 Token 预算不同。
def predict_caption(model, processor, image_path: str | Path, prompt: str = PROMPT, max_new_tokens: int = 128) -> str:
    import torch
    with Image.open(image_path) as im:
        image = im.convert("RGB")
    # 消息中声明图片占位和文字指令；实际图片在调用 processor 时传入。
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": prompt},
    ]}]
    # 按模型的对话模板组织文本，并添加供模型续写回答的起始标记。
    prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # 将文字和图片转换为模型输入张量，并移到 GPU。
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to("cuda")
    # 切换到评估模式，并关闭梯度记录；这里只推理，不更新权重。
    model.eval()
    # 关闭梯度追踪
    with torch.inference_mode(): 
        # 不进行随机采样，按当前生成配置生成最多 max_new_tokens 个文本 Token。
        ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Decoder-only 生成式架构，其返回序列包含输入前缀；切掉前缀，只解码新生成的回答。
    new_ids = ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_ids, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0].strip()

# 基线保存在本地数据目录，已有文件时优先读取，避免重复推理。
baseline_path = DATA_DIR/'baseline_prediction.json'
if baseline_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    # 核对验证图片、参考描述和推理条件；不一致时停止，避免混用实验结果。
    assert Path(baseline['image']).resolve() == Path(sample['image']).resolve(), '保存的基线图片与验证集不一致'
    assert baseline['reference_caption'] == sample['caption']
    assert baseline['prompt'] == PROMPT
    assert baseline.get('do_sample', False) is False
    assert baseline.get('max_new_tokens',128) == 128
    assert baseline.get('preprocessing_max_tokens',settings.image_max_tokens) == settings.image_max_tokens
    print('已复用基线预测:',baseline_path)
else:
    # PeftModel 是 PEFT 包装后的模型类；新基线必须由未挂载 LoRA 的模型生成。
    assert not isinstance(model,PeftModel), '基线必须是未挂载 LoRA 的模型'
    predicted = predict_caption(model,processor,sample['image'],PROMPT,128)
    # 同时记录参考答案、基础模型输出及实验条件，供微调后同图对比使用。
    baseline = {'image_id':sample['id'],'image':sample['image'],'prompt':PROMPT,
                'reference_caption':sample['caption'],'baseline_caption':predicted,
                'max_new_tokens':128,'do_sample':False,'preprocessing_max_tokens':settings.image_max_tokens}
    baseline_path.write_text(json.dumps(baseline,ensure_ascii=False,indent=2),encoding='utf-8')
print('GPT 参考 Caption:',baseline['reference_caption'])
print('Qwen2-VL 基线:',baseline['baseline_caption'])

## Step 6 · QLoRA：参数高效微调与 Adapter 防重复挂载

仅对 Q/K/V/O Attention Linear 层插入 LoRA：`r=8`、`alpha=16`、`dropout=0.05`；基础模型权重冻结。旧 Adapter 已存在时不会再次挂载未经训练的新 Adapter。

In [ ]:
# PEFT 配置、训练标签、优化器循环：全部定义在此 Notebook 内
import random
# * 后的 rank 只能用关键字传入，表示 LoRA 低秩矩阵的秩。
def attach_lora_once(base_model, *, rank: int = 8):
    from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
    # 防止在已经包装为 PEFT 模型的对象上重复添加 LoRA。
    if isinstance(base_model, PeftModel):
        raise ValueError("LoRA is already attached. Do not call get_peft_model twice.")
    # 冻结基础权重并准备量化模型训练；梯度检查点以额外计算换取更少的激活显存。
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    # 训练时关闭生成用的 KV 缓存，配合梯度检查点使用。
    base_model.config.use_cache = False
    # alpha 控制 LoRA 更新的缩放，dropout 用于正则化；仅适配指定的注意力投影层。
    config = LoraConfig(
        r=rank, lora_alpha=rank * 2, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none", task_type="CAUSAL_LM",
    )
    # 添加可训练的 LoRA 参数，打印其数量；此处尚未执行训练。
    model = get_peft_model(base_model, config)
    model.print_trainable_parameters()
    return model


def encode_training_sample(record: dict, processor, prompt: str = PROMPT):
    """Teacher-forcing: mask user/image prefix to -100 and train caption tokens."""
    import torch
    with Image.open(record["image"]) as im:
        image = im.convert("RGB")
    # 训练时将参考 Caption 作为 Assistant 答案，构造完整的监督对话。
    user = {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}
    assistant = {"role": "assistant", "content": str(record["caption"])}
    # 单独编码用户前缀，用于定位答案起点；完整文本则包含参考答案及结束标记。
    prefix_text = processor.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template([user, assistant], tokenize=False, add_generation_prompt=False)
    full = processor(text=[full_text], images=[image], return_tensors="pt")
    prefix = processor(text=[prefix_text], images=[image], return_tensors="pt")
    prefix_len = prefix["input_ids"].shape[1]
    # 确保单独编码的前缀与完整输入的开头一致，否则不能安全地按长度屏蔽标签。
    if not torch.equal(full["input_ids"][:, :prefix_len], prefix["input_ids"]):
        raise ValueError("Tokenizer prefix mismatch: verify chat template / special tokens")
    labels = full["input_ids"].clone()
    # -100 是损失函数的忽略标记：保留前缀作为上下文，但不要求模型预测这些位置。
    # 只监督答案部分及模板结束标记；因果语言模型会在内部对齐下一 Token 的预测目标。
    labels[:, :prefix_len] = -100
    if not (labels != -100).any():
        raise ValueError("No supervised caption tokens")
    full["labels"] = labels
    # 将输入张量和监督标签一起移到 GPU，供 model(**batch) 使用。
    return {key: tensor.to("cuda") for key, tensor in full.items()}


# 每次处理一张图片，累积多个样本的梯度后更新参数；返回各样本的原始 Loss。
# max_steps 限制优化器更新次数，不是样本数；0 表示不额外限制。
def train(records: list[dict], model, processor, *, epochs: int = 1,
          accumulation: int = 4, learning_rate: float = 1e-4, seed: int = 20260520,
          max_steps: int = 0) -> list[float]:
    import torch
    if not records:
        raise ValueError("Training set is empty")
    if accumulation < 1 or epochs < 1:
        raise ValueError("epochs and accumulation must be >= 1")
    # 优化器只接收 requires_grad=True 的参数；在当前配置中主要是新增的 LoRA 权重。
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=learning_rate)
    history = []
    optimizer.zero_grad(set_to_none=True)
    model.train()
    optimizer_updates = 0
    for epoch in range(epochs):
        # 不改变原始列表；每轮用不同但可复现的种子打乱顺序。
        shuffled = records.copy()
        random.Random(seed + epoch).shuffle(shuffled)
        for step, record in enumerate(shuffled, 1):
            batch = encode_training_sample(record, processor)
            # **batch 将字典展开为关键字参数；提供 labels 后模型会计算监督 Loss。
            output = model(**batch)
            loss = output.loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"NaN/Inf training loss at epoch={epoch+1} sample={step}")
            history.append(float(loss.item()))
            # 按当前累积窗口的实际样本数缩放 Loss，最后不足 accumulation 个样本也正确取平均。
            start = ((step - 1) // accumulation) * accumulation
            window = min(accumulation, len(shuffled) - start)
            (loss / window).backward()
            # 达到窗口边界或本轮最后一个样本时，才真正更新一次权重。
            if step % accumulation == 0 or step == len(shuffled):
                # 将可训练参数的整体梯度范数裁剪到 1.0；返回值是裁剪前的范数。
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                if not torch.isfinite(grad_norm):
                    raise RuntimeError("Invalid LoRA gradients")
                optimizer.step()
                # 清空已使用的梯度，避免带入下一个累积窗口。
                optimizer.zero_grad(set_to_none=True)
                optimizer_updates += 1
            print(f"Epoch {epoch+1}/{epochs} sample {step}/{len(shuffled)} loss={loss.item():.4f}")
            # 删除本次前向计算的引用；这不等于清空 PyTorch 的显存缓存。
            del batch, loss, output
            if max_steps and optimizer_updates >= max_steps:
                break
        # 内层 break 只退出样本循环，这里同时退出外层 Epoch 循环。
        if max_steps and optimizer_updates >= max_steps:
            break
    print(f"Optimizer updates: {optimizer_updates}; average loss={sum(history)/len(history):.4f}")
    return history

# 已保存的 Adapter 留待后续步骤加载；文件存在时不再添加一个新的未训练 Adapter。
ADAPTER_DIR = settings.lora_output_dir
adapter_exists = (ADAPTER_DIR/'adapter_config.json').is_file()
if adapter_exists:
    print('已存在 Adapter:',ADAPTER_DIR,'；下一步训练将默认跳过')
elif isinstance(model,PeftModel):
    print('当前 model 已挂载 LoRA；不会重复 get_peft_model')
else:
    # 仅添加 LoRA 结构；实际参数更新需在后续调用 train() 时执行。
    model = attach_lora_once(model,rank=8)

## Step 7 · 监督标签检查：只对 Assistant Caption 计算 Loss

图像 Token、用户 Prompt 和分隔 Token 不应进入监督目标，`labels[:prefix_len]=-100`。运行下面 Cell 可打印真实监督的文本。

In [ ]:
if adapter_exists:
    print('有现成 Adapter，本阶段可跳过监督标签验证')
else:
    batch = encode_training_sample(train_records[0],processor,prompt=PROMPT)
    print('input_ids:',tuple(batch['input_ids'].shape),'labels:',tuple(batch['labels'].shape))
    print('参与 Loss 的 Token:',int((batch['labels']!=-100).sum()))
    print('监督文字:',processor.decode(batch['labels'][0][batch['labels'][0]!=-100],skip_special_tokens=True))
    del batch

## Step 8 · 单样本 Forward / Loss / Backward（不更新权重）

单样本检查默认关闭，需要时打开 `RUN_GRADIENT_SMOKE_TEST`，观察 LoRA 是否存在非零有限梯度；不执行 `optimizer.step()`。

In [ ]:
RUN_GRADIENT_SMOKE_TEST = False
if RUN_GRADIENT_SMOKE_TEST:
    assert not adapter_exists and isinstance(model,PeftModel)
    model.config.use_cache = False
    model.train(); model.zero_grad(set_to_none=True)
    one_batch = encode_training_sample(train_records[0],processor,PROMPT)
    outputs = model(**one_batch)
    loss = outputs.loss
    assert torch.isfinite(loss), 'Loss 包含 NaN/Inf'
    loss.backward()
    grads = [p.grad for name,p in model.named_parameters()
             if 'lora_' in name and p.requires_grad and p.grad is not None]
    assert grads and all(torch.isfinite(g).all() for g in grads),'LoRA 梯度无效'
    norm = torch.sqrt(sum(g.detach().float().square().sum() for g in grads))
    assert norm.item()>0
    print('Loss=',loss.item(),'Gradient norm=',norm.item())
    model.zero_grad(set_to_none=True)
    del one_batch,outputs,loss,grads
else:
    print('梯度检查已跳过')

## Step 9 · 使用 19 张图片执行 QLoRA，并保存 Adapter

GPU 训练会产生费用。需要时手动将 `RUN_TRAINING=True`；已有 `adapter_config.json` 时禁止覆盖。一次 19 张、1 Epoch、累积 4，约 5 次优化器更新。

In [ ]:
RUN_TRAINING = False
loss_history = None
if RUN_TRAINING:
    assert not adapter_exists, f'现有 Adapter 不允许覆盖: {ADAPTER_DIR}'
    assert isinstance(model,PeftModel), '请先挂载 LoRA'
    loss_history = train(train_records,model,processor,epochs=1,accumulation=4,
                         learning_rate=1e-4,seed=settings.seed)
    ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(str(ADAPTER_DIR))
    processor.save_pretrained(str(ADAPTER_DIR))
    (ADAPTER_DIR/'train_summary.json').write_text(json.dumps({
        'model_id':settings.model_id,'prompt':PROMPT,'loss_history':loss_history,
        'train_samples':len(train_records),'epochs':1,'accumulation':4,'learning_rate':1e-4
    },ensure_ascii=False,indent=2),encoding='utf-8')
    adapter_exists = True
    print('LoRA 已保存:',ADAPTER_DIR)
else:
    print('未触发新训练；可直接用之前保存的 Adapter 推理。')

In [ ]:
# 查看训练 Loss 曲线（如果本次没有训练，尝试从上一次记录恢复）
if loss_history is None and (ADAPTER_DIR/'train_summary.json').is_file():
    loss_history = json.loads((ADAPTER_DIR/'train_summary.json').read_text(encoding='utf-8')).get('loss_history')
if loss_history:
    print('Loss 首值:',round(loss_history[0],4),'末值:',round(loss_history[-1],4))
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8,3)); plt.plot(range(1,len(loss_history)+1),loss_history,marker='o')
        plt.xlabel('Training sample step'); plt.ylabel('Cross-entropy Loss')
        plt.title('Qwen2-VL QLoRA Loss'); plt.grid(alpha=.3); plt.show()
    except ImportError:
        print('未安装 matplotlib，只展示文字 Loss。')
else:
    print('没有 Loss 历史记录')

## Step 10 · 同图、同 Prompt 微调后推理与结果对比

只能在保存 LoRA Adapter 后执行；仅比较输出风格/内容，不能证明对新场景泛化。

In [ ]:
adapter_exists = (ADAPTER_DIR/'adapter_config.json').is_file()
assert adapter_exists, '没有 LoRA Adapter：请先完成训练或使用已保存的 Adapter。'
assert Path(baseline['image']).resolve()==Path(sample['image']).resolve()
assert baseline['reference_caption']==sample['caption']
assert baseline.get('max_new_tokens',128)==128
assert baseline.get('do_sample',False) is False
assert baseline.get('preprocessing_max_tokens',settings.image_max_tokens)==settings.image_max_tokens
if not isinstance(model,PeftModel):
    model = PeftModel.from_pretrained(model,str(ADAPTER_DIR),is_trainable=False)
    print('已从云盘恢复 LoRA Adapter')
else:
    print('使用当前 Kernel 中的 LoRA Adapter')
finetuned_caption = predict_caption(model,processor,sample['image'],baseline['prompt'],128)
comparison = {**baseline,'finetuned_caption':finetuned_caption,
              'disclaimer':'Adjacent-frame qualitative smoke test, not generalization evidence'}
comparison_path = DATA_DIR/'inference_comparison.json'
comparison_path.write_text(json.dumps(comparison,ensure_ascii=False,indent=2),encoding='utf-8')
print('1. GPT 参考:',baseline['reference_caption'])
print('2. Base 模型:',baseline['baseline_caption'])
print('3. LoRA 模型:',finetuned_caption)
print('已保存结果:',comparison_path)

### 实际完成的 DSW 演示结果（独立于当前是否运行推理）

**GPT 参考**：A city street approaches a signalized intersection with marked crosswalks, cars ahead, and pedestrians on both sidewalks. Multi-story brick buildings line the road under a cloudy daytime sky.

**微调前**：输出建筑、车辆、行人、天气等 7 条项目列表，最后在 128 Token 截断。

**微调后**：A busy urban street with brick buildings on both sides, traffic lights, pedestrians, and vehicles, with a cloudy sky overhead.

结论：输出风格明显变简洁；仅 19/1 相似连续帧不能证明泛化改进。

In [ ]:
with Image.open(sample['image']) as im:
    display(im.convert('RGB').resize((800,450)))
if 'comparison' in globals():
    display(pd.DataFrame([
        {'模型':'GPT 参考','Caption':baseline['reference_caption']},
        {'模型':'Base Qwen2-VL','Caption':baseline['baseline_caption']},
        {'模型':'Qwen2-VL + LoRA','Caption':comparison['finetuned_caption']},
    ]))
else:
    print('尚未运行 Step 10，当前未生成三组对比结果。')

## ✅ Part 2 完成

已有产物位置：
- `vlm_training/images/`、`train.jsonl`、`validation.jsonl`、`split_manifest.json`
- `vlm_training/baseline_prediction.json`、`inference_comparison.json`
- `qwen2vl_lora/adapter_config.json`、LoRA 权重、可选的 `train_summary.json`

Notebook 本身不依赖业务 Python 模块，也不包含任何真实密钥、图像和权重。